In [1]:
import numpy as np

np.random.seed(42)

# Generate a synthetic regression dataset with 3 input features
X = np.random.uniform(-2, 2, (400, 3))

# Target is a nonlinear combination of the inputs
y = (
    np.sin(X[:, 0]) +
    0.5 * (X[:, 1] ** 2) -
    0.8 * X[:, 2]
)

y = y.reshape(-1, 1)

print("Input shape:", X.shape)
print("Target shape:", y.shape)


Input shape: (400, 3)
Target shape: (400, 1)


In [2]:
def relu(z):
    return np.maximum(0, z)

def relu_deriv(z):
    return (z > 0).astype(float)


def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_deriv(z):
    s = sigmoid(z)
    return s * (1 - s)


def tanh_act(z):
    return np.tanh(z)

def tanh_act_deriv(z):
    return 1 - np.tanh(z) ** 2


def leaky_relu(z, alpha=0.01):
    return np.where(z > 0, z, alpha * z)

def leaky_relu_deriv(z, alpha=0.01):
    return np.where(z > 0, 1, alpha)


def softplus(z):
    return np.log(1 + np.exp(z))

def softplus_deriv(z):
    return sigmoid(z)


In [3]:
def init_weights(layer_sizes):
    params = {}
    for l in range(1, len(layer_sizes)):
        params["W" + str(l)] = np.random.uniform(
            -0.5, 0.5, (layer_sizes[l], layer_sizes[l-1])
        )
        params["b" + str(l)] = np.zeros((layer_sizes[l], 1))
    return params


In [4]:
def forward_pass(X, params, act_fn):
    cache = {}
    A = X.T
    cache["A0"] = A
    num_layers = len(params) // 2

    # Hidden layers use the chosen activation
    for l in range(1, num_layers):
        W = params["W" + str(l)]
        b = params["b" + str(l)]
        Z = W @ A + b
        A = act_fn(Z)
        cache["Z" + str(l)] = Z
        cache["A" + str(l)] = A

    # Output layer is linear (regression)
    W_out = params["W" + str(num_layers)]
    b_out = params["b" + str(num_layers)]
    Z_out = W_out @ A + b_out
    A_out = Z_out
    cache["Z" + str(num_layers)] = Z_out
    cache["A" + str(num_layers)] = A_out
    return A_out, cache


In [5]:
def mean_squared_error(y_true, y_pred):
    return np.mean((y_pred.T - y_true) ** 2)


In [6]:
def backprop(X, y, params, cache, act_deriv):
    grads = {}
    batch_size = X.shape[0]
    num_layers = len(params) // 2
    y_pred = cache["A" + str(num_layers)]

    # Gradient of MSE with respect to output
    dA = 2 * (y_pred - y.T)

    for l in reversed(range(1, num_layers + 1)):
        Z = cache["Z" + str(l)]
        A_prev = cache["A" + str(l - 1)]
        if l == num_layers:
            dZ = dA  # linear output, no activation derivative needed
        else:
            dZ = dA * act_deriv(Z)
        grads["dW" + str(l)] = (1 / batch_size) * (dZ @ A_prev.T)
        grads["db" + str(l)] = (1 / batch_size) * np.sum(dZ, axis=1, keepdims=True)
        if l > 1:
            W = params["W" + str(l)]
            dA = W.T @ dZ
    return grads


In [7]:
def compute_grad_norm(g):
    return np.sqrt(np.sum(g ** 2))


In [8]:
def run_experiment(layer_sizes, act_fn, act_deriv_fn):
    params = init_weights(layer_sizes)
    learning_rate = 0.01
    num_epochs = 1000
    loss_at_200 = None
    loss_history = []

    for epoch in range(num_epochs):
        y_pred, cache = forward_pass(X, params, act_fn)
        current_loss = mean_squared_error(y, y_pred)
        loss_history.append(current_loss)

        if epoch == 200:
            loss_at_200 = current_loss

        grads = backprop(X, y, params, cache, act_deriv_fn)

        for l in range(1, len(layer_sizes)):
            params["W" + str(l)] -= learning_rate * grads["dW" + str(l)]
            params["b" + str(l)] -= learning_rate * grads["db" + str(l)]

    # Gradient norms at the first and last weight layers
    norm_first = compute_grad_norm(grads["dW1"])
    norm_last  = compute_grad_norm(grads["dW" + str(len(layer_sizes) - 1)])

    # Classify convergence behaviour
    if loss_history[-1] < loss_history[0] * 0.1:
        status = "Stable"
    elif loss_history[-1] < loss_history[0]:
        status = "Slow"
    else:
        status = "Unstable"

    return loss_history[-1], loss_at_200, norm_first, norm_last, status


In [9]:
# Architectures to compare: from shallow to very deep
architectures = {
    "A": [3, 4, 1],
    "B": [3, 6, 6, 1],
    "C": [3, 8, 8, 8, 8, 1],
    "D": [3, 8, 8, 8, 8, 8, 8, 8, 8, 1]
}


In [10]:
records = []

activation_map = {
    "ReLU":    (relu, relu_deriv),
    "Sigmoid": (sigmoid, sigmoid_deriv)
}

for arch_name, layer_sizes in architectures.items():
    for act_name in activation_map:
        fn, fn_deriv = activation_map[act_name]
        final_loss, l200, gn1, gnL, convergence = run_experiment(layer_sizes, fn, fn_deriv)
        records.append([arch_name, act_name, final_loss, l200, gn1, gnL, convergence])


In [11]:
import pandas as pd

results_df = pd.DataFrame(
    records,
    columns=[
        "Model",
        "Activation",
        "Final Loss",
        "Loss @200",
        "Grad Norm L1",
        "Grad Norm Last",
        "Observation"
    ]
)

results_df


,Model,Activation,Final Loss,Loss @200,Grad Norm L1,Grad Norm Last,Observation
0,A,ReLU,0.111545,0.492686,0.045217,0.037273,Stable
1,A,Sigmoid,0.415810,1.367625,0.037874,0.063916,Slow
2,B,ReLU,0.094230,1.313780,0.055468,0.041026,Stable
3,B,Sigmoid,0.999787,1.699662,0.228174,0.267797,Slow
4,C,ReLU,0.057146,0.505147,0.028671,0.004873,Stable
5,C,Sigmoid,1.740906,1.741983,0.004393,0.007336,Slow
6,D,ReLU,0.077302,1.737661,0.047787,0.009650,Stable
7,D,Sigmoid,1.743853,1.743853,0.000003,0.000001,Slow


In [12]:
## Reflections

# Did more depth consistently speed up convergence?
# Not always -- some deeper networks converged no faster and occasionally slower than shallower ones.

# Did gradient magnitudes remain consistent across all layers?
# No -- gradients in earlier layers tended to be noticeably smaller than those closer to the output.

# Was training equally stable across both activation functions?
# No -- sigmoid led to much slower learning as network depth increased.

# Which activation function handled depth better?
# ReLU maintained stable and effective training across all architectures tested.

# Did any networks progress slowly despite using the same learning rate?
# Yes -- deeper networks with sigmoid activation barely improved regardless of epoch count.
